# AI-Powered Google Drive Organizer

## Objective

Design an AI-based system that automatically organizes files in a Google Drive by analyzing their content and context. The system should itself create different category-wise folders in Google Drive and move files into them without manual intervention.

## Problem Statement

You are given a Google Drive containing multiple unorganized files (PDFs, Docs, Sheets, Images, etc.). Using AI, build an organizer that -

- Understands the context of each file
- Automatically creates separate folders (e.g., HR, Finance, Academics, Projects, Marketing, Personal)
- Moves files into the appropriate category folder in Google Drive

## Expected Capabilities

- Read file content and metadata (file name, type, text)
- Use AI/NLP to classify files by context
- Auto-create folders on Google Drive
- Auto-move files to the correct folder
- Handle uncertain cases gracefully

## Deliverables

- Short explanation of system design
- AI approach used for classification
- Tools/technologies proposed
- Sample workflow (1 example)
- Limitations & future improvements

## Bonus (Optional)

- Flowchart or pseudocode
- Real-time auto-organization on file upload

## Evaluation Criteria

Clarity, AI understanding, feasibility, and logical folder organization.

<hr>

### 1. Setting up the environment

- <b><i>google-auth-oauthlib -</i></b> To handle the "Login with Google" popup window.

- <b><i>google-api-python-client -</i></b> The official tool to send commands to Drive (move, create, delete).

- <b><i>google-genai -</i></b> To access Gemini (the AI brain).

- <b><i>PyPDF2 -</i></b> A lightweight tool to open PDF files and read text from them.

In [1]:
!pip install google-genai google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client PyPDF2

In [2]:
!pip install Pillow 
# To Handle image processing

### 2. Imports and configurations

In [3]:
import os.path 
import io # To handle file streams (like downloading a file to memory, not disk)

# GOOGLE AUTHENTICATION LIBRARIES -
from google.auth.transport.requests import Request # Handles refreshing expired tokens automatically

from google.oauth2.credentials import Credentials # Stores user access tokens

from google_auth_oauthlib.flow import InstalledAppFlow # Launches the local server for login

from googleapiclient.discovery import build # "Builds" the service object we use to call the API

from googleapiclient.errors import HttpError # Helps us catch specific Google API errors (like 404 Not Found)

from googleapiclient.http import MediaIoBaseDownload # Specialized tool to download file content from Drive

# AI and PDF LIBRARIES -
from google import genai
import PyPDF2
from PIL import Image

# CONFIGURATION -
GEMINI_API_KEY = ""

# The specific folder we want to organize.
TARGET_FOLDER_ID = "16KpzhwmD5ndJkTliTwjE9NsVE60HRWwF"

# The buckets we want the AI to sort files into
CATEGORIES = ["Finance", "HR", "Projects", "Personal", "Marketing", "Academics"]

# The permission level we need. 
# 'drive' = Full access (read, write, delete, move).
SCOPES = ['https://www.googleapis.com/auth/drive']

print("Libraries imported and configuration loaded.")

Libraries imported and configuration loaded.


### 3. Authentication

In [4]:
def authenticate_drive():
    """
        Authenticates the user with Google Drive API.
        Returns a 'service' object that we can use to make API calls.
    """
    creds = None
    
    # Check if we have a valid login token saved from last time
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
        
    # If no valid token, log in required
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            # If token expired, refresh it silently (no login popup needed)
            creds.refresh(Request())
        else:
            # If no token at all, open the browser popup
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES) # Requires credentials.json from Cloud Console
            creds = flow.run_local_server(port = 0)
            
        # Save the new token for next time
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    # Build and return the Drive Service
    return build('drive', 'v3', credentials = creds)

# Initialize the service immediately to test connection
service = authenticate_drive()
print("Authentication successful! Connected to Google Drive.")

Authentication successful! Connected to Google Drive.


### 4. File Reader and Helper Function

In [5]:
def read_file_content(service, file_id, mime_type):
    """
        Downloads file content. 
            - Returns TEXT string for Docs/PDFs.
            - Returns IMAGE object for JPG/PNG.
    """
    try:
        # CASE A - Images (JPG/PNG)
        if 'image/' in mime_type:
            request = service.files().get_media(fileId = file_id)
            fh = io.BytesIO()
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while done is False:
                status, done = downloader.next_chunk()
            
            # Open as an image object
            fh.seek(0)
            return Image.open(fh)

        # CASE B - Google Docs (Export to text)
        elif mime_type == 'application/vnd.google-apps.document':
            request = service.files().export_media(fileId = file_id, mimeType = 'text/plain')
            fh = io.BytesIO()
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while done is False:
                status, done = downloader.next_chunk()
            return fh.getvalue().decode('utf-8')[:2000]

        # CASE C - PDFs and Text Files
        elif 'pdf' in mime_type or 'text' in mime_type:
            request = service.files().get_media(fileId = file_id)
            fh = io.BytesIO()
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while done is False:
                status, done = downloader.next_chunk()
            
            fh.seek(0)
            
            if 'pdf' in mime_type:
                try:
                    reader = PyPDF2.PdfReader(fh)
                    content = ""
                    for i in range(min(2, len(reader.pages))):
                        content += reader.pages[i].extract_text()
                    return content
                except:
                    return ""
            else:
                return fh.read().decode('utf-8')[:2000]

        return None # Unsupported type

    except Exception as e:
        print(f"[!] Error reading content -> {e}")
        return None

def create_folder_if_not_exists(service, folder_name, parent_id):
    """
        Checks if a category folder (e.g., 'Finance') exists inside the Target Folder.
        If yes -> Returns its ID.
        If no -> Creates it and returns the new ID.
    """
    # Query - Look for a folder with THIS name inside THIS parent
    query = f"name = '{folder_name}' and '{parent_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    results = service.files().list(q = query, spaces = 'drive', fields = 'files(id, name)').execute()
    files = results.get('files', [])

    if files:
        return files[0]['id'] # Found it!
    else:
        # Not found, let's create it
        file_metadata = {
            'name': folder_name,
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': [parent_id]
        }
        file = service.files().create(body = file_metadata, fields = 'id').execute()
        print(f"Created new folder: {folder_name}")
        return file.get('id')

def move_file(service, file_id, new_parent_id, dry_run = False):
    """
        Actually moves the file by updating its 'parents' property.
    """
    if dry_run:
        print(f"[DRY RUN] Would move file {file_id} to {new_parent_id}")
        return

    try:
        # Get current parents (so we can remove them)
        file = service.files().get(fileId = file_id, fields = 'parents').execute()
        previous_parents = ",".join(file.get('parents'))
        
        # Update the file: Add new parent, Remove old parent
        service.files().update(
            fileId = file_id,
            addParents = new_parent_id,
            removeParents = previous_parents,
            fields = 'id, parents'
        ).execute()
    except HttpError as error:
        print(f"Error moving file: {error}")

print("Helper functions loaded.")

Helper functions loaded.


### 5. AI Classifier

In [6]:
def classify_file_with_content(filename, mime_type, content_data):
    """
        Handles both TEXT and IMAGE inputs for classification.
    """
    client = genai.Client(api_key = GEMINI_API_KEY)
    
    # Base instruction
    instruction = (
        f"I have a file named '{filename}' (Type: {mime_type}). "
        f"Categorize it into strictly one of these folders: {CATEGORIES}. "
        f"Reply ONLY with the category name. If unsure, reply 'Misc'."
    )
    
    try:
        # Check if content is an Image (JPG/PNG) or Text
        if isinstance(content_data, Image.Image):
            # MULTIMODAL MODE -> We send the Image Object + Text Instruction
            response = client.models.generate_content(
                model = 'gemini-2.5-flash',
                contents = [content_data, instruction]
            )
        else:
            # TEXT MODE -> We just send the text prompt
            if not content_data: 
                content_data = "No readable content."
                
            prompt = f"{instruction}\n\nFile Content Snippet:\n\"\"\"{content_data}\"\"\""
            
            response = client.models.generate_content(
                model = 'gemini-2.5-flash',
                contents = prompt
            )
        
        category = response.text.strip()
        
        if category not in CATEGORIES:
            return "Misc"
        return category

    except Exception as e:
        print(f"AI Error: {e}")
        return "Misc"

### 6. Execution Loop

In [7]:
import time

# SAFETY SWITCH -
# True = Simulation Mode (Prints what it WOULD do)
# False = Real Mode (Actually moves files)
DRY_RUN = True 

print(f"Scanning Drive Folder ID: {TARGET_FOLDER_ID}...")
print(f"Mode: {'SAFE MODE (Simulation)' if DRY_RUN else 'LIVE MODE (Moving Files)'}")

# List all files in the target folder (exclude folders, include only files)
query = f"'{TARGET_FOLDER_ID}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = false"
results = service.files().list(q = query, pageSize = 20, fields = "nextPageToken, files(id, name, mimeType)").execute()
items = results.get('files', [])

if not items:
    print('No files found to organize.')
else:
    print(f"Found {len(items)} files. Starting organization...\n")

    for item in items:
        name = item['name']
        file_id = item['id']
        mime = item['mimeType']
        
        print(f"Processing: {name}")
        
        # READ CONTENT -
        # We peek inside the file to give the AI more context
        print("...reading file content...")
        content_snippet = read_file_content(service, file_id, mime)
        
        # AI CLASSIFICATION -
        category = "Misc"
        retries = 0
        
        while retries < 3:
            category = classify_file_with_content(name, mime, content_snippet)
            
            # If we got "Misc" instantly, it might be an error or just unsure. 
            # In a real app, we'd check the exact error code, but for now:
            if category != "Misc":
                break # Success!
            
            # If it failed/defaulted to Misc, we wait and retry just in case it was a blip
            break 
        
        print(f"AI Decision: {category}")
        
        # EXECUTE MOVE -
        dest_folder_id = create_folder_if_not_exists(service, category, TARGET_FOLDER_ID)
        
        if dest_folder_id:
            move_file(service, file_id, dest_folder_id, dry_run = DRY_RUN)
            if not DRY_RUN:
                print(f"Moved to {category}")
        
        print("-" * 40) # Separator line

        # PAUSING FOR 60 SECONDS BETWEEN FILES
        # This keeps us within the free tier limits.
        print("Cooling down for 60 seconds to respect API limits...")
        time.sleep(60)

Scanning Drive Folder ID: 16KpzhwmD5ndJkTliTwjE9NsVE60HRWwF...
Mode: SAFE MODE (Simulation)
Found 6 files. Starting organization...

Processing: Public-Finance-in-Markets_simple.png
...reading file content...
AI Decision: Finance
Created new folder: Finance
[DRY RUN] Would move file 1QHx1Hn3lptWhYNY07SU5JayLBv6ShLDD to 15jAE4gg8mh0Go-SJaqFO3IcvCtryYOD0
----------------------------------------
Cooling down for 60 seconds to respect API limits...
Processing: Tradefinance_final_rev_02-e456c914d9eb47a9ac069c396dd0f09b.png
...reading file content...
AI Decision: Finance
[DRY RUN] Would move file 16Wq5Xl6hPVHQknUF_t5VwS5Hl3l9VM3v to 15jAE4gg8mh0Go-SJaqFO3IcvCtryYOD0
----------------------------------------
Cooling down for 60 seconds to respect API limits...
Processing: Social_Media_Poster.jpg
...reading file content...
AI Decision: Marketing
Created new folder: Marketing
[DRY RUN] Would move file 1l_tPb3CXyggurVkVjic6NdRqigEDZ3Mm to 1-fgnMcQ5Oxvk0YLtwWDot9w542-afEuw
------------------------